# Faza 2 — AI-SPEAK preprocessing po VIPL demo pipeline-u


## Goal

Pretvori `spk*/ser/video_a/*.mp4` u numerisane `128×64` mouth JPEG frejmove
koristeći brzi BlazeFace detector, isti 68-point face alignment, VIPL afinu
transformaciju i crop. Ne generiše se manifest, vocab, split ili statičan ROI.
Neuspesi ostaju u običnom logu.


## Setup

Izaberi T4 GPU. ZIP se jednom kopira sa Drive-a na `/content`; landmark obrada
sa montiranog Drive-a bi bila znatno sporija. Prilagodi samo `ZIP_ON_DRIVE`.


In [ ]:
import subprocess, sys
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'face-alignment==1.4.1', 'editdistance>=0.8.1'],
    check=True,
)


In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/nikolabakic/Vizuelno-prepoznavanje-govora-na-osnovu-pokreta-usana-pomo-u-LipNet-modela.git'
REPO = Path('/content/lipnet-serbian')
if not (REPO / 'lipnet').exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
os.chdir(REPO)
print('Repo:', REPO)
print('Commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())


In [ ]:
import numpy as np
import torch
assert torch.cuda.is_available(), 'Uključi T4 GPU u Colab Runtime postavkama.'
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
from google.colab import drive

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/LipNet')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print('Drive izlaz:', DRIVE_ROOT)


## Steps

### 1. Kopiraj i raspakuj lokalni korpus


In [ ]:
import shutil
import zipfile

ZIP_ON_DRIVE = Path('/content/drive/MyDrive/processed.zip')  # promeni po potrebi
LOCAL_ZIP = Path('/content/processed.zip')
EXTRACT_ROOT = Path('/content/ai_speak_source')
OUTPUT_ROOT = Path('/content/ai_speak_lip_blazeface')
CHECKPOINT_DIR = DRIVE_ROOT / 'phase2_chunks_blazeface'
RUN_FULL_PREPROCESSING = True

assert ZIP_ON_DRIVE.exists(), ZIP_ON_DRIVE
if not LOCAL_ZIP.exists() or not zipfile.is_zipfile(LOCAL_ZIP):
    if LOCAL_ZIP.exists():
        LOCAL_ZIP.unlink()
    shutil.copy2(ZIP_ON_DRIVE, LOCAL_ZIP)

EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
alignment = next(EXTRACT_ROOT.rglob('spk*/alignment/*.align'), None)
if alignment is None:
    print('AI-SPEAK sadržaj nije pronađen; raspakujem processed.zip...')
    with zipfile.ZipFile(LOCAL_ZIP) as archive:
        archive.extractall(EXTRACT_ROOT)
    alignment = next(EXTRACT_ROOT.rglob('spk*/alignment/*.align'), None)

assert alignment is not None, (
    f'Posle raspakivanja nema spk*/alignment/*.align u {EXTRACT_ROOT}. '
    'Proveri strukturu processed.zip arhive.'
)
CORPUS_ROOT = alignment.parents[2]
videos = list(CORPUS_ROOT.glob('spk*/ser/video_a/*.mp4'))
annotations = list(CORPUS_ROOT.glob('spk*/alignment/*.align'))
assert videos and len(videos) == len(annotations), (len(videos), len(annotations))
print('Korpus:', CORPUS_ROOT)
print('MP4/ALIGN:', len(videos), len(annotations))

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print('Drive checkpoint folder:', CHECKPOINT_DIR)


### 2. Prvo uradi jedan GPU smoke primer i upstream `_load_vid` proveru


In [ ]:
SMOKE_ROOT = Path('/content/ai_speak_lip_blazeface_smoke')
smoke_command = [
    sys.executable, '-m', 'scripts.prepare_ai_speak',
    '--corpus', str(CORPUS_ROOT), '--output', str(SMOKE_ROOT),
    '--device', 'cuda', '--face-detector', 'blazeface', '--limit', '1',
]
print('Pokrećem:', ' '.join(smoke_command))
smoke = subprocess.run(smoke_command, text=True, capture_output=True)
print(smoke.stdout)
if smoke.returncode:
    print(smoke.stderr)
    raise RuntimeError(
        f'AI-SPEAK smoke preprocessing nije uspeo (exit={smoke.returncode}). '
        'Stvarna greška je odštampana neposredno iznad.'
    )

from lipnet.dataset import MyDataset
sample_folder = next(SMOKE_ROOT.glob('spk*/video/video_a/*'))
sample_array = MyDataset._load_vid(sample_folder)
normalized = sample_array / 255.0
assert sample_array.shape[1:] == (64, 128, 3)
assert 0.0 <= float(normalized.min()) <= float(normalized.max()) <= 1.0
print('VIPL _load_vid:', sample_array.shape, 'range:', normalized.min(), normalized.max())


### 3. Obradi ceo korpus na GPU-u i nastavi bez ponavljanja gotovih klipova


In [ ]:
if RUN_FULL_PREPROCESSING:
    subprocess.run([
        sys.executable, '-m', 'scripts.prepare_ai_speak',
        '--corpus', str(CORPUS_ROOT), '--output', str(OUTPUT_ROOT),
        '--device', 'cuda', '--face-detector', 'blazeface', '--resume',
        '--report-every', '5',
        '--checkpoint-dir', str(CHECKPOINT_DIR),
        '--checkpoint-every', '10',
    ], check=True)
else:
    print('RUN_FULL_PREPROCESSING=False: ceo GPU posao je namerno preskočen.')


## Checks

### 4. Proveri potpunost i vizuelno pregledaj granične klipove


In [ ]:
import json
from IPython.display import Image, display

QA_PATH = OUTPUT_ROOT / 'qa_mouth_crops.jpg'
FAILURE_LOG = OUTPUT_ROOT / 'failed_clips.log'
PREPROCESSING_LOG = OUTPUT_ROOT / 'preprocessing.jsonl'
assert QA_PATH.exists(), 'Puna obrada nije napravljena ili nije završena.'
records = [
    json.loads(line)
    for line in PREPROCESSING_LOG.read_text(encoding='utf-8').splitlines()
    if line.strip()
]
failures = [
    line for line in FAILURE_LOG.read_text(encoding='utf-8').splitlines()
    if line.strip()
]
success_ids = {record['sample_id'] for record in records}
failure_ids = {line.split('\t', 1)[0] for line in failures}
input_ids = {path.stem for path in videos}
assert not (success_ids & failure_ids)
assert success_ids | failure_ids == input_ids
assert len(records) == len(success_ids)
assert all(record['landmark_frames'] > 0 for record in records)
assert all(
    record['decoded_frames'] == record['landmark_frames'] + record['dropped_frames']
    for record in records
)
phase2_audit = {
    'phase': 2,
    'face_detector': 'blazeface',
    'torch_version': torch.__version__,
    'gpu': torch.cuda.get_device_name(0),
    'input_pairs': len(videos),
    'successful_clips': len(records),
    'failed_clips': len(failures),
    'clips_with_dropped_frames': sum(
        record['dropped_frames'] > 0 for record in records
    ),
    'decoded_frames': sum(record['decoded_frames'] for record in records),
    'landmark_frames': sum(record['landmark_frames'] for record in records),
}
(OUTPUT_ROOT / 'phase2_audit.json').write_text(
    json.dumps(phase2_audit, indent=2, ensure_ascii=False) + '\n',
    encoding='utf-8',
)
display(Image(filename=str(QA_PATH)))
print('Uspešni klipovi:', len(records), '/', len(videos))
print('Neuspeli klipovi:', len(failures))
print('Klipovi sa ispuštenim frejmovima:', sum(r['dropped_frames'] > 0 for r in records))
print('\n'.join(failures[:20]) if failures else 'Nema neuspelih klipova.')
print('Ručno potvrdi: usne su u centru, nisu odsečene i crop je stabilan.')


### 5. Arhiviraj JPEG foldere i logove na Drive


In [ ]:
MANUAL_QA_PASSED = False  # promeni na True tek nakon pregleda prikazane QA slike
assert MANUAL_QA_PASSED, 'Ručno pregledaj QA sliku, pa potvrdi MANUAL_QA_PASSED=True.'
if RUN_FULL_PREPROCESSING:
    archive_base = Path('/content/ai_speak_lip')
    archive = Path(shutil.make_archive(str(archive_base), 'zip', root_dir=OUTPUT_ROOT))
    drive_archive = DRIVE_ROOT / 'ai_speak_lip.zip'
    shutil.copy2(archive, drive_archive)
    print('Sačuvano:', drive_archive, f'{drive_archive.stat().st_size/1024**3:.2f} GiB')


## Next Steps

Ne prelazi na Dataset dok smoke shape/opseg i ručni QA ne prođu. Faza 3 čita samo
JPEG foldere i originalne `.align` fajlove; `preprocessing.jsonl` i failure log nisu
ulaz u trening.
